# Radiomics Classification — Papilledema vs Normal
## Final Submission Notebook

**Binary classification:** Normal = 0, Papilledema = 1  
**Patient-level strict cross-validation — no patient leakage**  
**Pipeline: Imputation → VarianceThreshold → Correlation Filter → RobustScaler → MRMR → Classifier**

## 1. Install Dependencies

In [ ]:
!pip install -q optuna mrmr-selection scikit-learn pandas numpy matplotlib seaborn

## 2. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import json
import glob
import shutil

from copy import deepcopy
from collections import defaultdict
from itertools import combinations

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               GradientBoostingClassifier, VotingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    balanced_accuracy_score, brier_score_loss, average_precision_score,
    roc_curve, auc, precision_recall_curve
)
from scipy.stats import friedmanchisquare, wilcoxon

from mrmr import mrmr_classif

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)

print('All imports successful.')
print('Optuna version:', optuna.__version__)


## 3. Upload Data Files

Upload `normal_radiomics.csv` and `papilodem_radiomics.csv` using the cell below.

In [ ]:
# ── Google Colab file upload ──────────────────────────────────────────────────
try:
    from google.colab import files
    print('Running in Google Colab. Please upload both CSV files when prompted.')
    uploaded = files.upload()  # Upload normal_radiomics.csv and papilodem_radiomics.csv
    NORMAL_PATH = 'normal_radiomics.csv'
    PAPILL_PATH = 'papilodem_radiomics.csv'
except ImportError:
    # Running locally — set paths accordingly
    NORMAL_PATH = 'normal_radiomics.csv'
    PAPILL_PATH = 'papilodem_radiomics.csv'
    print('Running locally. Using default paths.')

## 4. Load and Merge Dataset

In [ ]:
df_normal = pd.read_csv(NORMAL_PATH)
df_normal['Label'] = 0  # Normal

df_papill = pd.read_csv(PAPILL_PATH)
df_papill['Label'] = 1  # Papilledema

df = pd.concat([df_normal, df_papill], ignore_index=True)

# Encode patient groups as consecutive integers
df['PatientIndex'] = pd.factorize(df['PatientIndex'])[0]

FEATURE_COLS = [c for c in df.columns if c.startswith('Feature_')]
print(f'Total samples   : {len(df)}')
print(f'Unique patients : {df["PatientIndex"].nunique()}')
print(f'Features        : {len(FEATURE_COLS)}')
print(f'Class balance   :')
print(df['Label'].value_counts().rename({0: 'Normal', 1: 'Papilledema'}).to_string())

## 5. Exploratory Data Analysis

In [ ]:
# ── Class distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df['Label'].value_counts().rename({0: 'Normal', 1: 'Papilledema'})
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Sample-Level Class Distribution', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(label_counts.index, rotation=0)
for i, v in enumerate(label_counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

patient_labels = df.groupby('PatientIndex')['Label'].max().value_counts().rename({0: 'Normal', 1: 'Papilledema'})
patient_labels.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], edgecolor='black')
axes[1].set_title('Patient-Level Class Distribution', fontsize=13)
axes[1].set_ylabel('Unique Patients')
axes[1].set_xticklabels(patient_labels.index, rotation=0)
for i, v in enumerate(patient_labels):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/class_distribution.png')

In [ ]:
# ── Images per patient distribution ────────────────────────────────────────
imgs_per_patient = df.groupby('PatientIndex').size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(imgs_per_patient.values, bins=20, color='steelblue', edgecolor='black')
ax.set_title('Images per Patient Distribution', fontsize=13)
ax.set_xlabel('Number of Images')
ax.set_ylabel('Number of Patients')
ax.axvline(imgs_per_patient.mean(), color='tomato', linestyle='--', label=f'Mean = {imgs_per_patient.mean():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/images_per_patient.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Mean images/patient: {imgs_per_patient.mean():.1f}')
print(f'Min: {imgs_per_patient.min()}, Max: {imgs_per_patient.max()}')

In [ ]:
# ── Missing value analysis ──────────────────────────────────────────────────
X_raw = df[FEATURE_COLS].copy()
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_pct = (X_raw.isna().sum() / len(X_raw) * 100)
print(f'Features with any missing: {(missing_pct > 0).sum()}')
print(f'Max missing % in any feature: {missing_pct.max():.2f}%')
print(f'Total NaN cells: {X_raw.isna().sum().sum()}')

## 6. Preprocessing Functions

All steps fit on training data only and transform validation/test data.

In [ ]:
MRMR_TOP_K = 15
VAR_THRESHOLD = 0.01
CORR_THRESHOLD = 0.95


def replace_inf_with_nan(X: pd.DataFrame) -> pd.DataFrame:
    """Replace ±Inf with NaN."""
    return X.replace([np.inf, -np.inf], np.nan)


def fit_median_imputer(X_train: pd.DataFrame) -> dict:
    """Fit median per feature on training data only."""
    return {'medians': X_train.median()}


def apply_median_imputer(X: pd.DataFrame, imputer: dict) -> pd.DataFrame:
    return X.fillna(imputer['medians'])


def fit_variance_filter(X_train: pd.DataFrame, threshold: float = VAR_THRESHOLD) -> dict:
    """Fit VarianceThreshold on training data only."""
    sel = VarianceThreshold(threshold=threshold)
    sel.fit(X_train.values)
    kept_cols = X_train.columns[sel.get_support()].tolist()
    return {'selector': sel, 'kept_cols': kept_cols}


def apply_variance_filter(X: pd.DataFrame, vf: dict) -> pd.DataFrame:
    return X[vf['kept_cols']]


def fit_correlation_filter(X_train: pd.DataFrame, threshold: float = CORR_THRESHOLD) -> dict:
    """Remove features with absolute Pearson correlation > threshold (fit on training only)."""
    corr_matrix = X_train.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    kept_cols = [c for c in X_train.columns if c not in to_drop]
    return {'kept_cols': kept_cols}


def apply_correlation_filter(X: pd.DataFrame, cf: dict) -> pd.DataFrame:
    return X[cf['kept_cols']]


def fit_robust_scaler(X_train: pd.DataFrame) -> dict:
    scaler = RobustScaler()
    scaler.fit(X_train.values)
    return {'scaler': scaler, 'cols': X_train.columns.tolist()}


def apply_robust_scaler(X: pd.DataFrame, rs: dict) -> pd.DataFrame:
    scaled = rs['scaler'].transform(X.values)
    return pd.DataFrame(scaled, columns=rs['cols'], index=X.index)


def fit_mrmr(X_train: pd.DataFrame, y_train: pd.Series, k: int = MRMR_TOP_K) -> dict:
    """Select top-k features using MRMR on training data only.
    Relevance  = Mutual Information (F-statistic)
    Redundancy = Mean Absolute Pearson Correlation
    Both match the assignment specification exactly.
    """
    # Reset both indices immediately before mrmr_classif to prevent
    # pandas.errors.IndexingError from misaligned split indices.
    X_train = X_train.reset_index(drop=True)
    y_train = pd.Series(y_train).reset_index(drop=True)
    selected = mrmr_classif(X=X_train, y=y_train, K=k)
    assert len(selected) == k, f'MRMR returned {len(selected)} features, expected {k}'
    return {'selected_features': selected}


def apply_mrmr(X: pd.DataFrame, mrmr_result: dict) -> pd.DataFrame:
    return X[mrmr_result['selected_features']]


def full_preprocess_fit(X_train: pd.DataFrame, y_train: pd.Series) -> dict:
    """Fit the complete preprocessing pipeline on training data ONLY.
    Order: Inf→NaN | Median Imputation | VarianceThreshold | Correlation Filter | RobustScaler | MRMR
    """
    X = replace_inf_with_nan(X_train.copy())

    imp = fit_median_imputer(X)
    X = apply_median_imputer(X, imp)

    vf = fit_variance_filter(X)
    X = apply_variance_filter(X, vf)

    cf = fit_correlation_filter(X)
    X = apply_correlation_filter(X, cf)

    rs = fit_robust_scaler(X)
    X = apply_robust_scaler(X, rs)

    y_reset = y_train.reset_index(drop=True) if hasattr(y_train, 'reset_index') else pd.Series(y_train)
    mrmr_res = fit_mrmr(X, y_reset)
    X = apply_mrmr(X, mrmr_res)

    pipeline = {
        'imputer':            imp,
        'variance_filter':    vf,
        'correlation_filter': cf,
        'robust_scaler':      rs,
        'mrmr':               mrmr_res
    }
    return pipeline, X


def full_preprocess_transform(X: pd.DataFrame, pipeline: dict) -> pd.DataFrame:
    """Apply a FITTED pipeline to new data (transform only — no fitting)."""
    X = replace_inf_with_nan(X.copy())
    X = apply_median_imputer(X, pipeline['imputer'])
    X = apply_variance_filter(X, pipeline['variance_filter'])
    X = apply_correlation_filter(X, pipeline['correlation_filter'])
    X = apply_robust_scaler(X, pipeline['robust_scaler'])
    X = apply_mrmr(X, pipeline['mrmr'])
    return X


print('Preprocessing pipeline functions defined.')
print('Pipeline order: Inf→NaN | Median Imputation | VarianceThreshold(0.01) | CorrFilter(0.95) | RobustScaler | MRMR(top-15)')


## 7. Cross-Validation Setup

In [ ]:
X_all = df[FEATURE_COLS].copy()
y_all = df['Label'].values
groups_all = df['PatientIndex'].values

N_OUTER_SPLITS = 5   # Patient-level stratified group k-fold
N_OPTUNA_TRIALS = 50
MRMR_TOP_K = 15

outer_cv = StratifiedGroupKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=SEED)

# Verify no patient leakage across folds
print('Verifying patient-level splits (no leakage):')
for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_all, y_all, groups_all)):
    train_patients = set(groups_all[train_idx])
    test_patients  = set(groups_all[test_idx])
    overlap = train_patients & test_patients
    print(f'  Fold {fold_idx+1}: train={len(train_patients)} patients, test={len(test_patients)} patients, overlap={len(overlap)}')
    assert len(overlap) == 0, f'Patient leakage detected in fold {fold_idx+1}!'
print('No patient leakage detected.')

## 8. Model Definitions and Optuna Hyperparameter Spaces

In [ ]:
def get_lr_params(trial):
    return {
        'C':           trial.suggest_float('C', 1e-3, 10.0, log=True),
        'solver':      trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
        'max_iter':    2000,
        'random_state': SEED
    }

def build_lr(params):
    return LogisticRegression(**params)


def get_svm_params(trial):
    return {
        'C':           trial.suggest_float('C', 1e-2, 100.0, log=True),
        'gamma':       trial.suggest_float('gamma', 1e-4, 1.0, log=True),
        'kernel':      'rbf',
        'probability': False,   # CalibratedClassifierCV provides probability output
        'random_state': SEED
    }

def build_svm(params):
    return SVC(**params)


def get_rf_params(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 400),
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      SEED
    }

def build_rf(params):
    return RandomForestClassifier(**params)


def get_et_params(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 400),
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      SEED
    }

def build_et(params):
    return ExtraTreesClassifier(**params)


def get_gb_params(trial):
    return {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state':      SEED
    }

def build_gb(params):
    return GradientBoostingClassifier(**params)


def get_knn_params(trial):
    return {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 20),
        'weights':     trial.suggest_categorical('weights', ['uniform', 'distance']),
        'metric':      trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski'])
    }

def build_knn(params):
    return KNeighborsClassifier(**params)


MODEL_REGISTRY = {
    'LogisticRegression': (get_lr_params,  build_lr),
    'RBF_SVM':            (get_svm_params, build_svm),
    'RandomForest':       (get_rf_params,  build_rf),
    'ExtraTrees':         (get_et_params,  build_et),
    'GradientBoosting':   (get_gb_params,  build_gb),
    'KNN':                (get_knn_params, build_knn),
}

print('Model registry defined:', list(MODEL_REGISTRY.keys()))


## 9. Optuna Objective — Inner CV with Patient-Level Groups

In [ ]:
def make_objective(model_name, X_train_df, y_train, groups_train, n_inner_splits=3):
    """
    Returns an Optuna objective function.
    Inner CV uses StratifiedGroupKFold to preserve patient grouping.
    Preprocessing is fitted ONLY on inner-training data inside each fold.
    """
    get_params_fn, build_fn = MODEL_REGISTRY[model_name]

    # Dynamic inner split guard — ensure enough patients per inner fold
    n_unique_patients = len(np.unique(groups_train))
    safe_inner_splits = min(n_inner_splits, n_unique_patients // 4)
    safe_inner_splits = max(safe_inner_splits, 2)

    def objective(trial):
        params = get_params_fn(trial)
        inner_cv = StratifiedGroupKFold(
            n_splits=safe_inner_splits, shuffle=True, random_state=SEED
        )

        fold_f1_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(X_train_df, y_train, groups_train):
            X_it = X_train_df.iloc[inner_train_idx]
            y_it = pd.Series(y_train).iloc[inner_train_idx].values
            X_iv = X_train_df.iloc[inner_val_idx]
            y_iv = pd.Series(y_train).iloc[inner_val_idx].values

            # Full preprocessing fitted on inner-training fold only
            pipeline, X_it_proc = full_preprocess_fit(X_it, pd.Series(y_it))
            X_iv_proc = full_preprocess_transform(X_iv, pipeline)

            clf = build_fn(params)
            clf.fit(X_it_proc.values, y_it)
            y_pred_inner = clf.predict(X_iv_proc.values)
            fold_f1_scores.append(
                f1_score(y_iv, y_pred_inner, average='macro', zero_division=0)
            )

        return np.mean(fold_f1_scores)

    return objective


print('Optuna objective function defined.')
print('Inner CV: StratifiedGroupKFold with dynamic n_splits guard.')
print('Objective: Macro-F1 | Sampler: TPE')


## 10. Main Training Loop — Outer CV with Optuna per Fold per Model

In [ ]:
# ── Storage ────────────────────────────────────────────────────────────────
all_results              = defaultdict(list)
best_models              = defaultdict(list)
best_params_per_model    = defaultdict(list)
selected_features_per_model = defaultdict(list)

X_all_df = X_all.copy()

print(f'Starting outer CV: {N_OUTER_SPLITS} folds x {len(MODEL_REGISTRY)} models x {N_OPTUNA_TRIALS} Optuna trials')
print('Metrics collected: Accuracy, Balanced Accuracy, Precision, Recall, Macro-F1, ROC-AUC, PR-AUC, Brier Score')
print('='*70)

for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_all_df, y_all, groups_all)):

    print(f'\n[FOLD {fold_idx+1}/{N_OUTER_SPLITS}] '
          f'Train: {len(train_idx)} samples | Test: {len(test_idx)} samples')

    X_train_raw = X_all_df.iloc[train_idx]
    y_train      = y_all[train_idx]
    g_train      = groups_all[train_idx]

    X_test_raw = X_all_df.iloc[test_idx]
    y_test      = y_all[test_idx]

    for model_name in MODEL_REGISTRY:
        print(f'  [{model_name}] Optuna search ({N_OPTUNA_TRIALS} trials)...', end=' ')

        objective = make_objective(
            model_name, X_train_raw, y_train, g_train, n_inner_splits=3
        )

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

        best_params = study.best_params
        best_params_per_model[model_name].append(best_params)

        # ── Refit on full outer training set with preprocessing ────────
        _, build_fn = MODEL_REGISTRY[model_name]
        pipeline, X_train_proc = full_preprocess_fit(X_train_raw, pd.Series(y_train))
        X_test_proc = full_preprocess_transform(X_test_raw, pipeline)

        selected_features_per_model[model_name].append(
            pipeline['mrmr']['selected_features']
        )

        base_clf = build_fn(best_params)

        # ── Sigmoid Calibration ───────────────────────────────────────
        cal_clf = CalibratedClassifierCV(base_clf, cv=3, method='sigmoid')
        cal_clf.fit(X_train_proc.values, y_train)

        y_pred = cal_clf.predict(X_test_proc.values)
        y_prob = cal_clf.predict_proba(X_test_proc.values)[:, 1]

        has_both_classes = len(np.unique(y_test)) > 1

        metrics = {
            'fold':               fold_idx + 1,
            'accuracy':           accuracy_score(y_test, y_pred),
            'balanced_accuracy':  balanced_accuracy_score(y_test, y_pred),
            'precision':          precision_score(y_test, y_pred, average='macro', zero_division=0),
            'recall':             recall_score(y_test, y_pred, average='macro', zero_division=0),
            'macro_f1':           f1_score(y_test, y_pred, average='macro', zero_division=0),
            'roc_auc':            roc_auc_score(y_test, y_prob) if has_both_classes else np.nan,
            'pr_auc':             average_precision_score(y_test, y_prob) if has_both_classes else np.nan,
            'brier_score':        brier_score_loss(y_test, y_prob),
            'best_trial_score':   study.best_value,
            'y_test':             y_test.tolist(),
            'y_pred':             y_pred.tolist(),
            'y_prob':             y_prob.tolist(),
        }
        all_results[model_name].append(metrics)
        best_models[model_name].append(cal_clf)

        print(f'inner_F1={study.best_value:.3f} | '
              f'test_F1={metrics["macro_f1"]:.3f} | '
              f'AUC={metrics["roc_auc"]:.3f} | '
              f'Brier={metrics["brier_score"]:.3f}')

print('\n' + '='*70)
print('Outer CV complete.')


## 11. Aggregate Results per Model

In [ ]:
summary_rows = []
for model_name, fold_results in all_results.items():
    accs   = [r['accuracy']           for r in fold_results]
    bacs   = [r['balanced_accuracy']  for r in fold_results]
    precs  = [r['precision']          for r in fold_results]
    recs   = [r['recall']             for r in fold_results]
    f1s    = [r['macro_f1']           for r in fold_results]
    aucs   = [r['roc_auc']            for r in fold_results if not np.isnan(r['roc_auc'])]
    prauc  = [r['pr_auc']             for r in fold_results if not np.isnan(r['pr_auc'])]
    briers = [r['brier_score']        for r in fold_results]

    summary_rows.append({
        'Model':             model_name,
        'Accuracy':          f'{np.mean(accs):.4f} +/- {np.std(accs):.4f}',
        'Balanced_Accuracy': f'{np.mean(bacs):.4f} +/- {np.std(bacs):.4f}',
        'Precision':         f'{np.mean(precs):.4f} +/- {np.std(precs):.4f}',
        'Recall':            f'{np.mean(recs):.4f} +/- {np.std(recs):.4f}',
        'Macro_F1':          f'{np.mean(f1s):.4f} +/- {np.std(f1s):.4f}',
        'ROC_AUC':           f'{np.mean(aucs):.4f} +/- {np.std(aucs):.4f}' if aucs else 'N/A',
        'PR_AUC':            f'{np.mean(prauc):.4f} +/- {np.std(prauc):.4f}' if prauc else 'N/A',
        'Brier_Score':       f'{np.mean(briers):.4f} +/- {np.std(briers):.4f}',
        '_mean_f1':          np.mean(f1s),
        '_mean_auc':         np.mean(aucs) if aucs else np.nan,
        '_mean_acc':         np.mean(accs),
        '_mean_bac':         np.mean(bacs),
    })

display_cols = ['Model', 'Accuracy', 'Balanced_Accuracy', 'Precision',
                'Recall', 'Macro_F1', 'ROC_AUC', 'PR_AUC', 'Brier_Score']

df_summary = (pd.DataFrame(summary_rows)
              .sort_values('_mean_f1', ascending=False)
              .reset_index(drop=True))

print('\n=== Model Performance Summary (Mean +/- Std across folds) ===')
print(df_summary[display_cols].to_string(index=False))

df_summary[display_cols].to_csv('results/tables/model_summary.csv', index=False)
print('\nSaved: results/tables/model_summary.csv')


## 12. Per-Fold Detailed Results

In [ ]:
per_fold_rows = []
for model_name, fold_results in all_results.items():
    for r in fold_results:
        per_fold_rows.append({
            'Model':     model_name,
            'Fold':      r['fold'],
            'Accuracy':  round(r['accuracy'],  4),
            'Precision': round(r['precision'], 4),
            'Recall':    round(r['recall'],    4),
            'Macro_F1':  round(r['macro_f1'],  4),
            'ROC_AUC':   round(r['roc_auc'],   4) if not np.isnan(r['roc_auc']) else np.nan,
        })

df_per_fold = pd.DataFrame(per_fold_rows)
print(df_per_fold.to_string(index=False))
df_per_fold.to_csv('results/tables/per_fold_results.csv', index=False)
print('\nSaved: results/tables/per_fold_results.csv')

## 13. Confusion Matrices per Model

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
class_names = ['Normal', 'Papilledema']

for ax_idx, model_name in enumerate(MODEL_REGISTRY.keys()):
    fold_results = all_results[model_name]

    # Aggregate predictions across all folds
    y_true_all = []
    y_pred_all = []
    for r in fold_results:
        y_true_all.extend(r['y_test'])
        y_pred_all.extend(r['y_pred'])

    cm = confusion_matrix(y_true_all, y_pred_all)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=axes[ax_idx], cbar=False
    )
    axes[ax_idx].set_title(f'{model_name}\n(Aggregated across folds)', fontsize=11)
    axes[ax_idx].set_xlabel('Predicted')
    axes[ax_idx].set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models (Aggregated CV Predictions)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/figures/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/confusion_matrices.png')

## 14. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(MODEL_REGISTRY)))

for (model_name, fold_results), color in zip(all_results.items(), colors):
    y_true_all = []
    y_prob_all = []
    for r in fold_results:
        y_true_all.extend(r['y_test'])
        y_prob_all.extend(r['y_prob'])

    fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{model_name} (AUC = {roc_auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models (Aggregated CV Predictions)', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/roc_curves.png')

## 15. Metric Comparison Bar Charts

In [ ]:
metric_cols   = ['_mean_acc', '_mean_f1', '_mean_auc']
metric_labels = ['Mean Accuracy', 'Mean Macro-F1', 'Mean ROC-AUC']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, label in zip(axes, metric_cols, metric_labels):
    vals   = df_summary[col].values
    models = df_summary['Model'].values
    colors_bar = ['#2196F3' if v != max(vals) else '#F44336' for v in vals]
    bars = ax.barh(models, vals, color=colors_bar, edgecolor='black')
    ax.set_xlim(0, 1.05)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Score')
    for bar, v in zip(bars, vals):
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Model Comparison — Key Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/metric_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/metric_comparison.png')


## 16. Per-Fold F1 Box Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

f1_data    = []
model_list = []
for model_name in MODEL_REGISTRY:
    f1s = [r['macro_f1'] for r in all_results[model_name]]
    f1_data.append(f1s)
    model_list.append(model_name)

bp = ax.boxplot(f1_data, labels=model_list, patch_artist=True, notch=False)
colors_box = plt.cm.Set2(np.linspace(0, 1, len(model_list)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)

ax.set_title('Macro-F1 Distribution across CV Folds', fontsize=13)
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/figures/f1_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/f1_boxplot.png')

## 17. Feature Stability Analysis (MRMR Selection Frequency)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for ax_idx, model_name in enumerate(MODEL_REGISTRY.keys()):
    feat_counter = defaultdict(int)
    for fold_features in selected_features_per_model[model_name]:
        for feat in fold_features:
            feat_counter[feat] += 1

    feat_df = pd.DataFrame.from_dict(feat_counter, orient='index', columns=['Count'])
    feat_df = feat_df.sort_values('Count', ascending=False).head(20)

    axes[ax_idx].barh(feat_df.index[::-1], feat_df['Count'][::-1], color='steelblue', edgecolor='black')
    axes[ax_idx].set_title(f'{model_name} — Top Feature Frequencies', fontsize=10)
    axes[ax_idx].set_xlabel('Selection Count (out of folds)')
    axes[ax_idx].axvline(N_OUTER_SPLITS * 0.5, color='red', linestyle='--', alpha=0.6, label='50% threshold')
    axes[ax_idx].legend(fontsize=8)

plt.suptitle('MRMR Feature Selection Stability across CV Folds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/feature_stability.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/feature_stability.png')

In [ ]:
# ── Save feature stability table ────────────────────────────────────────────
stability_rows = []
for model_name in MODEL_REGISTRY:
    feat_counter = defaultdict(int)
    for fold_features in selected_features_per_model[model_name]:
        for feat in fold_features:
            feat_counter[feat] += 1
    for feat, count in sorted(feat_counter.items(), key=lambda x: -x[1]):
        stability_rows.append({
            'Model':    model_name,
            'Feature':  feat,
            'Count':    count,
            'Pct':      round(count / N_OUTER_SPLITS * 100, 1)
        })

df_stability = pd.DataFrame(stability_rows)
df_stability.to_csv('results/tables/feature_stability.csv', index=False)
print('Saved: results/tables/feature_stability.csv')
print(df_stability[df_stability['Pct'] >= 80].to_string(index=False))

## 18. Classification Reports per Model

In [ ]:
report_rows = []
sep = '=' * 50

for model_name in MODEL_REGISTRY:
    y_true_all = []
    y_pred_all = []
    for r in all_results[model_name]:
        y_true_all.extend(r['y_test'])
        y_pred_all.extend(r['y_pred'])

    print(f'\n{sep}')
    print(f'Classification Report — {model_name}')
    print(sep)
    report_str = classification_report(
        y_true_all, y_pred_all,
        target_names=['Normal', 'Papilledema'],
        zero_division=0
    )
    print(report_str)

    with open(f'results/tables/classification_report_{model_name}.txt', 'w') as fh:
        fh.write(f'Classification Report — {model_name}\n{sep}\n')
        fh.write(report_str)

    report_dict = classification_report(
        y_true_all, y_pred_all,
        target_names=['Normal', 'Papilledema'],
        output_dict=True, zero_division=0
    )
    report_rows.append({
        'Model':            model_name,
        'Normal_Precision': round(report_dict['Normal']['precision'],      4),
        'Normal_Recall':    round(report_dict['Normal']['recall'],         4),
        'Normal_F1':        round(report_dict['Normal']['f1-score'],       4),
        'Papill_Precision': round(report_dict['Papilledema']['precision'], 4),
        'Papill_Recall':    round(report_dict['Papilledema']['recall'],    4),
        'Papill_F1':        round(report_dict['Papilledema']['f1-score'],  4),
        'Macro_F1':         round(report_dict['macro avg']['f1-score'],    4),
    })

df_reports = pd.DataFrame(report_rows)
df_reports.to_csv('results/tables/classification_reports_all.csv', index=False)
print('\nSaved: results/tables/classification_reports_all.csv')


## 19. Calibration Curves

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax_idx, model_name in enumerate(MODEL_REGISTRY.keys()):
    y_true_all = []
    y_prob_all = []
    for r in all_results[model_name]:
        y_true_all.extend(r['y_test'])
        y_prob_all.extend(r['y_prob'])

    try:
        frac_pos, mean_pred = calibration_curve(
            y_true_all, y_prob_all, n_bins=10, strategy='uniform'
        )
        axes[ax_idx].plot(mean_pred, frac_pos, 's-', color='steelblue', label='Calibrated model')
    except Exception:
        axes[ax_idx].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')

    axes[ax_idx].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[ax_idx].set_title(f'{model_name}', fontsize=10)
    axes[ax_idx].set_xlabel('Mean Predicted Probability')
    axes[ax_idx].set_ylabel('Fraction of Positives')
    axes[ax_idx].legend(fontsize=8)
    axes[ax_idx].grid(alpha=0.3)

plt.suptitle('Calibration Curves — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/calibration_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/calibration_curves.png')

## 20. Precision-Recall Curves

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(9, 7))
colors_pr = plt.cm.tab10(np.linspace(0, 1, len(MODEL_REGISTRY)))

for (model_name, fold_results), color in zip(all_results.items(), colors_pr):
    y_true_all = []
    y_prob_all = []
    for r in fold_results:
        y_true_all.extend(r['y_test'])
        y_prob_all.extend(r['y_prob'])

    precision_vals, recall_vals, _ = precision_recall_curve(y_true_all, y_prob_all)
    ap = average_precision_score(y_true_all, y_prob_all)
    ax.plot(recall_vals, precision_vals, color=color, lw=2,
            label=f'{model_name} (AP = {ap:.3f})')

baseline = sum(y_all) / len(y_all)
ax.axhline(baseline, color='gray', linestyle='--', label=f'Baseline (prevalence={baseline:.2f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13)
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/precision_recall_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/precision_recall_curves.png')

## 21. Ensemble Model (Soft Voting of Calibrated Models)

In [ ]:
# ── Ensemble: RF + ET + GB soft-voting (as per assignment specification) ───
ENSEMBLE_MODELS = ['RandomForest', 'ExtraTrees', 'GradientBoosting']
ensemble_results = []

print('Building Fixed Soft-Voting Ensemble: RF + ET + GB')
print('Reusing calibrated models from outer CV — no test-set tuning.')
print('='*60)

for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_all_df, y_all, groups_all)):

    X_test_raw = X_all_df.iloc[test_idx]
    y_test      = y_all[test_idx]
    X_train_raw = X_all_df.iloc[train_idx]
    y_train_ens = y_all[train_idx]

    # Refit pipeline on this fold's training data for test transform
    pipeline_ens, _ = full_preprocess_fit(X_train_raw, pd.Series(y_train_ens))
    X_test_proc = full_preprocess_transform(X_test_raw, pipeline_ens)

    # Reuse already-calibrated models from the main loop
    # Re-transform test using each model's own pipeline for correctness
    fold_probs = []
    for model_name in ENSEMBLE_MODELS:
        cal_clf = best_models[model_name][fold_idx]
        # Get this model's pipeline features for test data
        feats = selected_features_per_model[model_name][fold_idx]
        X_test_model = X_test_proc.reindex(columns=feats, fill_value=0)
        prob = cal_clf.predict_proba(X_test_model.values)[:, 1]
        fold_probs.append(prob)

    # Soft voting = mean of calibrated probabilities
    y_prob_ens = np.mean(fold_probs, axis=0)
    y_pred_ens = (y_prob_ens >= 0.5).astype(int)

    has_both = len(np.unique(y_test)) > 1
    ensemble_results.append({
        'fold':             fold_idx + 1,
        'accuracy':         accuracy_score(y_test, y_pred_ens),
        'balanced_accuracy':balanced_accuracy_score(y_test, y_pred_ens),
        'precision':        precision_score(y_test, y_pred_ens, average='macro', zero_division=0),
        'recall':           recall_score(y_test, y_pred_ens, average='macro', zero_division=0),
        'macro_f1':         f1_score(y_test, y_pred_ens, average='macro', zero_division=0),
        'roc_auc':          roc_auc_score(y_test, y_prob_ens) if has_both else np.nan,
        'pr_auc':           average_precision_score(y_test, y_prob_ens) if has_both else np.nan,
        'brier_score':      brier_score_loss(y_test, y_prob_ens),
        'y_test':           y_test.tolist(),
        'y_pred':           y_pred_ens.tolist(),
        'y_prob':           y_prob_ens.tolist(),
    })
    r = ensemble_results[-1]
    print(f'Fold {fold_idx+1}: F1={r["macro_f1"]:.4f} | AUC={r["roc_auc"]:.4f} | Brier={r["brier_score"]:.4f}')

print('\nEnsemble (RF+ET+GB) training complete.')


In [ ]:
ens_f1s   = [r['macro_f1'] for r in ensemble_results]
ens_aucs  = [r['roc_auc']  for r in ensemble_results if not np.isnan(r['roc_auc'])]
ens_accs  = [r['accuracy'] for r in ensemble_results]

print('\n=== Soft-Voting Ensemble Results ===')
print(f'  Accuracy   : {np.mean(ens_accs):.4f} ± {np.std(ens_accs):.4f}')
print(f'  Macro-F1   : {np.mean(ens_f1s):.4f} ± {np.std(ens_f1s):.4f}')
print(f'  ROC-AUC    : {np.mean(ens_aucs):.4f} ± {np.std(ens_aucs):.4f}')

ens_summary = pd.DataFrame([{
    'Model':     'Ensemble (SoftVoting)',
    'Accuracy':  f'{np.mean(ens_accs):.4f} ± {np.std(ens_accs):.4f}',
    'Macro_F1':  f'{np.mean(ens_f1s):.4f} ± {np.std(ens_f1s):.4f}',
    'ROC_AUC':   f'{np.mean(ens_aucs):.4f} ± {np.std(ens_aucs):.4f}',
}])
ens_summary.to_csv('results/tables/ensemble_summary.csv', index=False)
print('Saved: results/tables/ensemble_summary.csv')

In [ ]:
# ── Ensemble confusion matrix ───────────────────────────────────────────────
y_true_ens_all = []
y_pred_ens_all = []
y_prob_ens_all = []
for r in ensemble_results:
    y_true_ens_all.extend(r['y_test'])
    y_pred_ens_all.extend(r['y_pred'])
    y_prob_ens_all.extend(r['y_prob'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_ens = confusion_matrix(y_true_ens_all, y_pred_ens_all)
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Papilledema'],
            yticklabels=['Normal', 'Papilledema'],
            ax=axes[0], cbar=False)
axes[0].set_title('Ensemble — Confusion Matrix', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

fpr_ens, tpr_ens, _ = roc_curve(y_true_ens_all, y_prob_ens_all)
auc_ens = auc(fpr_ens, tpr_ens)
axes[1].plot(fpr_ens, tpr_ens, 'b-', lw=2, label=f'Ensemble (AUC = {auc_ens:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Ensemble — ROC Curve', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/ensemble_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/ensemble_evaluation.png')

print('\nEnsemble Classification Report:')
print(classification_report(y_true_ens_all, y_pred_ens_all,
                            target_names=['Normal', 'Papilledema'], zero_division=0))

## 22. Final Summary Table — All Models + Ensemble

In [ ]:
final_summary_rows = []

for model_name in MODEL_REGISTRY:
    fold_results = all_results[model_name]
    accs   = [r['accuracy']          for r in fold_results]
    bacs   = [r['balanced_accuracy'] for r in fold_results]
    precs  = [r['precision']         for r in fold_results]
    recs   = [r['recall']            for r in fold_results]
    f1s    = [r['macro_f1']          for r in fold_results]
    aucs   = [r['roc_auc']           for r in fold_results if not np.isnan(r['roc_auc'])]
    prauc  = [r['pr_auc']            for r in fold_results if not np.isnan(r['pr_auc'])]
    briers = [r['brier_score']       for r in fold_results]

    final_summary_rows.append({
        'Model':              model_name,
        'Type':               'Individual',
        'Accuracy':           round(np.mean(accs),   4),
        'Balanced_Accuracy':  round(np.mean(bacs),   4),
        'Precision':          round(np.mean(precs),  4),
        'Recall':             round(np.mean(recs),   4),
        'Macro_F1':           round(np.mean(f1s),    4),
        'ROC_AUC':            round(np.mean(aucs),   4) if aucs else np.nan,
        'PR_AUC':             round(np.mean(prauc),  4) if prauc else np.nan,
        'Brier_Score':        round(np.mean(briers), 4),
    })

ens_accs   = [r['accuracy']          for r in ensemble_results]
ens_bacs   = [r['balanced_accuracy'] for r in ensemble_results]
ens_precs  = [r['precision']         for r in ensemble_results]
ens_recs   = [r['recall']            for r in ensemble_results]
ens_f1s    = [r['macro_f1']          for r in ensemble_results]
ens_aucs   = [r['roc_auc']           for r in ensemble_results if not np.isnan(r['roc_auc'])]
ens_prauc  = [r['pr_auc']            for r in ensemble_results if not np.isnan(r['pr_auc'])]
ens_briers = [r['brier_score']       for r in ensemble_results]

final_summary_rows.append({
    'Model':              'Ensemble (RF+ET+GB)',
    'Type':               'Ensemble',
    'Accuracy':           round(np.mean(ens_accs),   4),
    'Balanced_Accuracy':  round(np.mean(ens_bacs),   4),
    'Precision':          round(np.mean(ens_precs),  4),
    'Recall':             round(np.mean(ens_recs),   4),
    'Macro_F1':           round(np.mean(ens_f1s),    4),
    'ROC_AUC':            round(np.mean(ens_aucs),   4) if ens_aucs else np.nan,
    'PR_AUC':             round(np.mean(ens_prauc),  4) if ens_prauc else np.nan,
    'Brier_Score':        round(np.mean(ens_briers), 4),
})

df_final = (pd.DataFrame(final_summary_rows)
            .sort_values('Macro_F1', ascending=False)
            .reset_index(drop=True))

df_final.to_csv('results/tables/final_summary_all_models.csv', index=False)

show_cols = ['Model', 'Type', 'Accuracy', 'Balanced_Accuracy',
             'Precision', 'Recall', 'Macro_F1', 'ROC_AUC', 'PR_AUC', 'Brier_Score']
print('=== FINAL SUMMARY — ALL MODELS + ENSEMBLE ===')
print(df_final[show_cols].to_string(index=False))
print('\nSaved: results/tables/final_summary_all_models.csv')


## 24. Statistical Tests — Friedman + Wilcoxon + Bonferroni

In [ ]:
# ── Friedman Test across all models ─────────────────────────────────────────
model_names_list = list(MODEL_REGISTRY.keys())
f1_matrix = np.array([
    [r['macro_f1'] for r in all_results[m]]
    for m in model_names_list
])  # shape: (n_models, n_folds)

stat_friedman, p_friedman = friedmanchisquare(*[f1_matrix[i] for i in range(len(model_names_list))])
print('=== Friedman Test ===')
print(f'Statistic : {stat_friedman:.4f}')
print(f'p-value   : {p_friedman:.4f}')
if p_friedman < 0.05:
    print('Result    : Significant differences between models (p < 0.05)')
else:
    print('Result    : No significant differences detected (p >= 0.05)')

with open('results/tables/friedman_test.txt', 'w') as fh:
    fh.write('Friedman Test\n')
    fh.write(f'Statistic: {stat_friedman:.4f}\n')
    fh.write(f'p-value: {p_friedman:.4f}\n')
    fh.write(f'n_models={len(model_names_list)}, n_folds={N_OUTER_SPLITS}\n')

# ── Pairwise Wilcoxon + Bonferroni Correction ─────────────────────────────
pairs          = list(combinations(range(len(model_names_list)), 2))
n_comparisons  = len(pairs)
wilcoxon_rows  = []

print('\n=== Pairwise Wilcoxon Signed-Rank Tests (Bonferroni, n=' + str(n_comparisons) + ') ===')
for i, j in pairs:
    scores_i = f1_matrix[i]
    scores_j = f1_matrix[j]
    diff = scores_i - scores_j
    if np.all(diff == 0):
        stat_w, p_w = np.nan, np.nan
        note = 'identical scores'
    else:
        try:
            stat_w, p_w = wilcoxon(scores_i, scores_j)
            note = ''
        except Exception as e:
            stat_w, p_w = np.nan, np.nan
            note = str(e)
    p_bonf = min(float(p_w) * n_comparisons, 1.0) if not np.isnan(p_w) else np.nan
    sig    = 'Yes' if (not np.isnan(p_bonf) and p_bonf < 0.05) else 'No'
    wilcoxon_rows.append({
        'Model_A':       model_names_list[i],
        'Model_B':       model_names_list[j],
        'Wilcoxon_stat': round(stat_w, 4) if not np.isnan(stat_w) else np.nan,
        'p_raw':         round(p_w,    4) if not np.isnan(p_w)    else np.nan,
        'p_bonferroni':  round(p_bonf, 4) if not np.isnan(p_bonf) else np.nan,
        'Significant':   sig,
        'Note':          note
    })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
print(df_wilcoxon.to_string(index=False))
df_wilcoxon.to_csv('results/tables/statistical_tests_wilcoxon.csv', index=False)
print('\nSaved: results/tables/friedman_test.txt')
print('Saved: results/tables/statistical_tests_wilcoxon.csv')


## 25. Feature Importance — Tree-Based Models

In [ ]:
# ── Feature Importance from tree-based models (mean across folds) ──────────
TREE_MODELS = ['RandomForest', 'ExtraTrees', 'GradientBoosting']
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, model_name in zip(axes, TREE_MODELS):
    importance_accum = defaultdict(list)

    for fold_idx, (train_idx, _) in enumerate(
            outer_cv.split(X_all_df, y_all, groups_all)):

        X_train_raw_fi = X_all_df.iloc[train_idx]
        y_train_fi     = y_all[train_idx]

        pipeline_fi, X_tr_proc_fi = full_preprocess_fit(
            X_train_raw_fi, pd.Series(y_train_fi)
        )

        _, build_fn = MODEL_REGISTRY[model_name]
        params_fi   = best_params_per_model[model_name][fold_idx]
        clf_fi      = build_fn(params_fi)
        clf_fi.fit(X_tr_proc_fi.values, y_train_fi)

        feats = pipeline_fi['mrmr']['selected_features']
        for feat, imp in zip(feats, clf_fi.feature_importances_):
            importance_accum[feat].append(imp)

    mean_imp = {f: np.mean(v) for f, v in importance_accum.items()}
    imp_df   = (pd.DataFrame.from_dict(mean_imp, orient='index', columns=['Importance'])
                .sort_values('Importance', ascending=False)
                .head(15))

    ax.barh(imp_df.index[::-1], imp_df['Importance'][::-1],
            color='steelblue', edgecolor='black')
    ax.set_title(f'{model_name}\nFeature Importance (Mean across {N_OUTER_SPLITS} folds)', fontsize=10)
    ax.set_xlabel('Mean Gini Importance')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance — Tree-Based Models (RF, ET, GB)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/feature_importance.png')


## 26. Final Summary Heatmap

In [ ]:
heatmap_df = df_final[['Model', 'Accuracy', 'Precision', 'Recall', 'Macro_F1', 'ROC_AUC']].set_index('Model')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    heatmap_df.astype(float),
    annot=True, fmt='.4f', cmap='YlOrRd',
    vmin=0.0, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax
)
ax.set_title('Model Performance Heatmap (Mean across CV Folds)', fontsize=13)
ax.set_xlabel('Metric')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('results/figures/performance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/figures/performance_heatmap.png')

## 27. List All Saved Files

In [ ]:
import glob

print('=== Saved Figures ===')
for f in sorted(glob.glob('results/figures/*.png')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

print('\n=== Saved Tables ===')
for f in sorted(glob.glob('results/tables/*.csv') + glob.glob('results/tables/*.txt')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

print('\n=== Pipeline Complete ===')

## 28. Download All Results (Colab)

In [ ]:
import shutil
shutil.make_archive('radiomics_results', 'zip', 'results')

try:
    from google.colab import files
    files.download('radiomics_results.zip')
    print('Download triggered: radiomics_results.zip')
except ImportError:
    print('Not in Colab. Results archive saved as: radiomics_results.zip')